# 向量存储记忆

> **将每轮对话嵌入为向量，在推理时检索语义最相关的 Top-K 历史轮次。为对话记忆引入语义搜索。**

想象一座没有目录的图书馆。要找任何东西，你得逐个书架地走。现在想象一位图书管理员，能立刻知道哪三本书最能回答你的问题。向量存储记忆为你的 Agent 配备了这样一位管理员——它按内容**含义**而非**时间**来查找上下文。

在之前的短期记忆 notebook 中，我们探索了基于近因的策略：保留最近 *k* 条消息（滑动窗口），或将旧轮次压缩为摘要。两者都依赖**何时**说的，而非**说了什么**。

**向量存储记忆**采用不同的方法。它将每轮对话转换为一个稠密嵌入向量（一串捕获含义的数字），并将其存入向量数据库。当 Agent 需要上下文时，当前查询被嵌入并与*所有*已存储轮次进行余弦相似度比较。Top-*K* 条语义最相关的片段被返回，无论它们发生在对话的哪个位置。

这与 RAG（检索增强生成）的核心思想相同，只是这里将其应用于 Agent 自身的对话历史，而非外部文档。

**收益：** 当用户在第 47 轮问到第 3 轮提及的事实时，Agent 能够回忆起来。滑动窗口早就丢弃了那条事实。

**完成本 notebook 后你将理解：**
- 如何使用 OpenAI 嵌入和 ChromaDB 从零构建向量存储记忆。
- 检索流水线：嵌入、存储、查询、注入提示词。
- 一个受控的 50 轮实验，证明语义召回在非顺序问题上大幅优于滑动窗口。
- 权衡：检索质量、成本、延迟，以及何时与其他记忆策略组合。

## 核心概念

- **嵌入模型 (Embedding model)**：将文本映射为固定长度向量的模型（如 OpenAI `text-embedding-3-small`），向量捕获文本语义。相似文本产生相似向量。
- **向量数据库 (Vector database)**：针对高维向量近似最近邻（ANN）搜索优化的存储。ANN 指“即使在海量数据中也能快速找到最接近匹配”。这里使用 **ChromaDB**（内存模式）。
- **余弦相似度 (Cosine similarity)**：比较向量的距离度量。余弦相似度越高，语义越相关。
- **Top-K 检索 (Top-K retrieval)**：返回与给定查询最相似的 *K* 条已存储向量。*K* 控制召回与成本的权衡。低 K 便宜但可能遗漏；高 K 覆盖更全但消耗更多 token。
- **分块策略 (Chunk strategy)**：如何为嵌入分割对话：单条消息、用户-助手对、或滑动窗口。此处嵌入**用户-助手对**以获得连贯的检索。
- **元数据 (Metadata)**：每条已存储向量的时间戳、轮次编号等附加属性，支持混合过滤（如“相关 *且* 最近的”）。
- **上下文注入 (Context injection)**：检索到的记忆被格式化并前置到提示词中，供 LLM 参考。

## 架构

<p align="center">
  <img src="../../images/diagrams/06_vector_store_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid 源码</summary>

```mermaid
flowchart LR
    subgraph Ingestion["摄入（每轮之后）"]
        A["用户消息 +
助手回复"] --> B["嵌入
模型"]
        B --> C["向量 +
元数据"]
        C --> D[("ChromaDB
（内存模式）")]
    end

    subgraph Retrieval["检索（每次 LLM 调用前）"]
        E["新的用户
查询"] --> F["嵌入
模型"]
        F --> G["余弦
相似度搜索"]
        D --> G
        G --> H["Top-K
相关轮次"]
    end

    subgraph Generation["生成"]
        H --> I["构建提示词：
系统提示 + 检索的
记忆 + 近期缓冲区"]
        I --> J["LLM
（Claude）"]
        J --> K["回复"]
    end

    style D fill:#4f46e5,color:#fff
    style J fill:#059669,color:#fff
```

</details>

In [ ]:
# 安装依赖（运行一次）
%pip install -q anthropic openai chromadb python-dotenv matplotlib numpy

加载环境变量并设置 API 客户端。需要 Anthropic key（Claude）和 OpenAI key（嵌入模型）。

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 API 密钥

import anthropic
import openai
import chromadb

assert os.getenv("ANTHROPIC_API_KEY"), "请在 .env 文件中设置 ANTHROPIC_API_KEY"
assert os.getenv("OPENAI_API_KEY"), "请在 .env 文件中设置 OPENAI_API_KEY（用于嵌入）"

print("\u2713 API 密钥已加载")
print(f"\u2713 ChromaDB 版本: {chromadb.__version__}")

## 核心实现

`VectorStoreMemory` 有三个职责：

1. **嵌入并存储** 每轮用户-助手交换。
2. **检索** 构建下一条提示词时最相关的 Top-K 历史交换。
3. **构建提示词** 将检索到的记忆与近期消息缓冲区组合。

使用的模型：
- **OpenAI `text-embedding-3-small`** 用于嵌入（1536 维，便宜、快速）。
- **ChromaDB** 内存 collection 用于向量存储和搜索。
- **Anthropic Claude** 用于对话模型。

In [ ]:
class VectorStoreMemory:
    """检索语义相关历史轮次的向量存储记忆。"""

    def __init__(
        self,
        top_k: int = 5,
        recent_buffer_size: int = 4,
        embedding_model: str = "text-embedding-3-small",
        chat_model: str = "claude-sonnet-4-20250514",
        system_prompt: str | None = None,
        max_tokens: int = 1024,
        collection_name: str = "conversation_memory",
    ):
        self.top_k = top_k
        self.recent_buffer_size = recent_buffer_size
        self.embedding_model = embedding_model
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens

        # API 客户端
        self.chat_client = anthropic.Anthropic()
        self.embed_client = openai.OpenAI()
        self.chat_model = chat_model

        # ChromaDB 内存 collection
        self.chroma = chromadb.Client()
        self.collection = self.chroma.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )

        # 状态
        self.turn_count = 0
        self.recent_buffer: list[dict] = []  # 最近 N 条消息（原文）
        self.full_history: list[dict] = []   # 完整日志，用于分析

        # 追踪
        self.turn_token_usage: list[dict] = []
        self.retrieval_log: list[dict] = []  # 每轮检索到的内容


接下来添加嵌入和存储方法。`_embed` 调用 OpenAI 嵌入 API 将文本转为向量。`_store_exchange` 将用户消息与助手回复组合、嵌入、并存入 ChromaDB（附带元数据）。

In [ ]:
    # -- 嵌入 -------------------------------------------------------
    def _embed(self, text: str) -> list[float]:
        """获取文本的嵌入向量。"""
        response = self.embed_client.embeddings.create(
            model=self.embedding_model,
            input=text,
        )
        return response.data[0].embedding

    # -- 存储 ------------------------------------------------------------
    def _store_exchange(self, user_msg: str, assistant_msg: str) -> None:
        """嵌入并存储一次用户-助手交换。"""
        self.turn_count += 1
        # 组合用户 + 助手以获得更丰富的嵌入
        combined = f"User: {user_msg}\nAssistant: {assistant_msg}"
        embedding = self._embed(combined)

        self.collection.add(
            ids=[f"turn-{self.turn_count}"],
            embeddings=[embedding],
            documents=[combined],
            metadatas=[{
                "turn": self.turn_count,
                "user_msg": user_msg,
                "assistant_msg": assistant_msg,
            }],
        )


接下来添加检索方法。当你发送新消息时，记忆会嵌入你的查询并在 ChromaDB 中搜索最接近的已存储交换。同时构建系统提示词，将检索到的记忆格式化为 LLM 可用的上下文。

In [ ]:
    # -- 检索 ---------------------------------------------------------
    def _retrieve(self, query: str, top_k: int | None = None) -> list[dict]:
        """检索 Top-K 条最相关的历史交换。"""
        k = top_k or self.top_k
        if self.turn_count == 0:
            return []

        query_embedding = self._embed(query)
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=min(k, self.turn_count),
        )

        retrieved = []
        for i in range(len(results["ids"][0])):
            retrieved.append({
                "turn": results["metadatas"][0][i]["turn"],
                "document": results["documents"][0][i],
                "distance": results["distances"][0][i] if results["distances"] else None,
            })
        return retrieved

    # -- 构建提示词 -----------------------------------------------------
    def _build_system_prompt(self, retrieved: list[dict]) -> str:
        """将系统提示词与检索到的记忆组合。"""
        parts = []
        if self.system_prompt:
            parts.append(self.system_prompt)

        if retrieved:
            memory_text = "\n\n".join(
                f"[Turn {r['turn']}] {r['document']}" for r in retrieved
            )
            parts.append(
                f"以下是本对话中相关的历史交换：\n\n"
                f"{memory_text}\n\n"
                f"在回复时请参考这些记忆。"
            )
        return "\n\n".join(parts) if parts else ""


`chat` 方法将所有环节串联起来。它检索相关历史轮次、构建提示词、调用 Claude、存储新交换、并更新近期缓冲区。这是每轮用户消息的主要入口。

In [ ]:
    # -- 对话 -------------------------------------------------------------
    def chat(self, user_input: str) -> str:
        """发送消息，检索相关上下文，获取回复。"""
        # 步骤 1：检索相关历史轮次
        retrieved = self._retrieve(user_input)
        self.retrieval_log.append({
            "turn": self.turn_count + 1,
            "query": user_input,
            "retrieved_turns": [r["turn"] for r in retrieved],
        })

        # 步骤 2：构建消息列表（近期缓冲区 + 新消息）
        messages = list(self.recent_buffer) + [{"role": "user", "content": user_input}]

        # 步骤 3：调用 LLM
        system = self._build_system_prompt(retrieved)
        kwargs = dict(
            model=self.chat_model,
            max_tokens=self.max_tokens,
            messages=messages,
        )
        if system:
            kwargs["system"] = system

        response = self.chat_client.messages.create(**kwargs)
        assistant_text = response.content[0].text

        # 步骤 4：将交换存入向量数据库
        self._store_exchange(user_input, assistant_text)

        # 步骤 5：更新近期缓冲区
        self.recent_buffer.append({"role": "user", "content": user_input})
        self.recent_buffer.append({"role": "assistant", "content": assistant_text})
        while len(self.recent_buffer) > self.recent_buffer_size:
            self.recent_buffer.pop(0)

        # 完整历史用于分析
        self.full_history.append({"role": "user", "content": user_input})
        self.full_history.append({"role": "assistant", "content": assistant_text})

        # 追踪 token
        self.turn_token_usage.append({
            "turn": self.turn_count,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "retrieved_count": len(retrieved),
        })

        return assistant_text


最后添加检查工具。`search` 方法允许直接查询向量存储以进行调试。`stats` 方法报告已存储的轮次数和向量数。

In [ ]:
    # -- 检查 -------------------------------------------------------
    def search(self, query: str, top_k: int = 5) -> list[dict]:
        """公开的搜索接口，用于调试记忆内容。"""
        return self._retrieve(query, top_k)

    def stats(self) -> dict:
        return {
            "total_turns": self.turn_count,
            "vectors_stored": self.collection.count(),
            "recent_buffer_size": len(self.recent_buffer),
        }

    def __repr__(self) -> str:
        return (
            f"VectorStoreMemory(turns={self.turn_count}, "
            f"vectors={self.collection.count()}, "
            f"top_k={self.top_k})"
        )


print("\u2713 VectorStoreMemory 类已定义")

## 使用示例：语义召回实战

让我们在不同轮次中植入多条事实，然后乱序提问。向量存储应该能检索到相关轮次，即使它发生在很多轮之前。

In [ ]:
mem = VectorStoreMemory(
    top_k=3,
    recent_buffer_size=4,
    system_prompt="你是一个简洁的助手。回复控制在 1-2 句话。",
    collection_name="usage_demo",
)

messages = [
    "我叫 Priya，是 Spotify 的数据科学家。",
    "我上个月领养了一只叫 Zephyr 的救援灰狗。",
    "我正在学习 Rust 来构建音乐推荐引擎。",
    "我最喜欢的菜系是埃塞俄比亚菜——我喜欢 injera 配 misir wot。",
    "我养了什么宠物？",              # 应该检索到第 2 轮
    "我在学什么编程语言？",          # 应该检索到第 3 轮
]

for msg in messages:
    print(f"\U0001f464 用户:  {msg}")
    reply = mem.chat(msg)
    print(f"\U0001f916 Agent: {reply}")
    print()

print(f"\n\U0001f4ca 统计: {mem.stats()}")

检查向量存储为每个查询检索到了什么。检索日志显示每条用户消息命中了哪些已存储轮次。这有助于验证语义搜索是否找到了正确的上下文。

In [ ]:
# 查看向量存储对每个查询检索到的内容
print("=== 检索日志 ===\n")
for entry in mem.retrieval_log:
    if entry["retrieved_turns"]:
        query_display = (
            f"{entry['query'][:50]}..." if len(entry["query"]) > 50
            else entry["query"]
        )
        print(f"第 {entry['turn']} 轮: \"{query_display}\"")
        print(f"  \u2192 检索到的轮次: {entry['retrieved_turns']}")
        print()

你也可以直接搜索记忆，像对话上的语义搜索引擎一样。这对调试很有用：传入一个查询，查看哪些已存储交换在含义上最接近。

In [ ]:
# 直接搜索记忆——就像对话上的语义搜索引擎
results = mem.search("食物偏好", top_k=3)
print("\U0001f50d 搜索: '食物偏好'\n")
for r in results:
    print(f"  第 {r['turn']} 轮 (距离: {r['distance']:.4f}):")
    print(f"    {r['document'][:100]}...")
    print()

## 实验：50 轮对话，向量存储 vs. 滑动窗口

这是核心实验。我们将：

1. 生成一段 **50 轮合成对话**，在特定轮次植入具体事实。
2. 完成全部 50 轮后，提出引用早期事实的**回忆问题**。
3. 将相同的问题分别在**向量存储记忆**（Top-K=5）和**滑动窗口**（K=10 条消息）上运行。
4. 比较回忆准确率：哪个系统记住更多？

假设：向量存储记忆能从对话的任意位置召回事实。滑动窗口只能回忆最近约 5 轮中的事实。

In [ ]:
# 50 轮合成对话：在特定轮次植入事实，中间用填充对话隔开。
# 每个条目为（轮次编号, 用户消息）。
# "事实"轮次植入可检索的具体信息。
# "填充"轮次是通用问题，用于将事实推出滑动窗口。

PLANTED_FACTS = {
    1:  ("我叫 Jordan Rivera，今年 34 岁。", "jordan rivera", "姓名"),
    3:  ("我在 Boston Dynamics 担任机器人工程师。", "boston dynamics", "雇主"),
    5:  ("我的血型是 AB 阴性——非常罕见。", "ab negative", "血型"),
    8:  ("我对贝类严重过敏——随身携带 EpiPen。", "shellfish", "过敏"),
    12: ("我童年养的狗叫 Biscuit，是只比格犬。", "biscuit", "童年宠物"),
    16: ("我去年跑了波士顿马拉松，成绩 3 小时 42 分。", "3 hours 42", "马拉松"),
    20: ("我家 Wi-Fi 密码是 'correct-horse-battery-staple'。", "correct-horse", "密码"),
    25: ("我有一个双胞胎姐姐叫 Avery，住在巴塞罗那。", "avery", "姐妹"),
    30: ("我最喜欢的书是 Hofstadter 的《哥德尔、埃舍尔、巴赫》。", "godel", "书"),
    35: ("我在马丘比丘山顶向我的伴侣求婚。", "machu picchu", "求婚"),
    40: ("我的车是 2019 款深绿色 Subaru Outback。", "subaru outback", "车"),
    45: ("我在 2021 年给大学室友捐了一个肾。", "kidney", "捐赠"),
}


定义填充消息，用于填充植入事实之间的轮次。这些是通用问题，会把植入的事实推出滑动窗口。在真实对话中，这就是重要时刻之间的闲聊。

In [ ]:
FILLER_MESSAGES = [
    "今天波士顿天气怎么样？",
    "能解释一下神经网络是怎么工作的吗？",
    "讲一个关于章鱼的有趣事实。",
    "蒙古的首都是什么？",
    "酸面包是怎么做的？",
    "TCP 和 UDP 有什么区别？",
    "讲讲詹姆斯·韦伯太空望远镜。",
    "CRISPR 基因编辑是怎么工作的？",
    "板球的规则是什么？",
    "解释一下蒙提霍尔问题。",
    "世界上最深的洞穴是哪个？",
    "潮汐是怎么形成的？",
    "讲讲互联网的历史。",
    "雷声是怎么产生的？",
    "飞机是怎么停留在空中的？",
    "地球上最大的沙漠是哪个？",
    "用简单的话解释区块链。",
    "北极光是什么？",
    "微波炉是怎么工作的？",
    "陆地上跑得最快的动物是什么？",
    "讲讲伏尼契手稿。",
    "海豚是怎么睡觉的？",
    "地震是怎么引起的？",
    "GPS 导航是怎么工作的？",
    "已知最古老的活树是什么？",
    "解释一下安慰剂效应。",
    "降噪耳机是怎么工作的？",
    "世界上最高的瀑布是哪个？",
    "讲讲费米悖论。",
    "冰箱是怎么工作的？",
    "四季是怎么形成的？",
    "触摸屏是怎么工作的？",
    "讲讲深海。",
    "暗物质是什么？",
    "信鸽是怎么导航的？",
    "彩虹是怎么形成的？",
    "疫苗如何触发免疫力？",
    "世界上使用人数最多的语言是什么？",
]


现在将植入事实和填充消息组装成 50 轮对话。同时定义回忆问题，每个问题针对一条特定事实，并包含一个用于在回答中检测的关键词。

In [ ]:
# 构建 50 轮对话
conversation_50 = []
filler_idx = 0
for turn in range(1, 51):
    if turn in PLANTED_FACTS:
        msg = PLANTED_FACTS[turn][0]
    else:
        msg = FILLER_MESSAGES[filler_idx % len(FILLER_MESSAGES)]
        filler_idx += 1
    conversation_50.append((turn, msg))

print(f"已构建合成对话: {len(conversation_50)} 轮")
print(f"植入事实的轮次: {sorted(PLANTED_FACTS.keys())}")

# 回忆问题——每个问题针对一条特定植入事实
RECALL_QUESTIONS = [
    ("我的全名和年龄是什么？",           "jordan rivera",    1),
    ("我在哪里工作？",                    "boston dynamics",   3),
    ("我的血型是什么？",                  "ab negative",      5),
    ("我对什么食物过敏？",                "shellfish",        8),
    ("我童年养的狗叫什么？",              "biscuit",          12),
    ("我的马拉松成绩是多少？",            "3 hours 42",       16),
    ("我家 Wi-Fi 密码是什么？",          "correct-horse",    20),
    ("我有兄弟姐妹吗？",                  "avery",            25),
    ("我最喜欢的书是什么？",              "godel",            30),
    ("我在哪里向伴侣求婚的？",            "machu picchu",     35),
    ("我开什么车？",                      "subaru",           40),
    ("我捐献过器官吗？",                  "kidney",           45),
]

print(f"回忆问题: {len(RECALL_QUESTIONS)} 个")

将全部 50 轮对话送入向量存储记忆。每轮被嵌入并存储。之后，记忆中包含 50 个可供检索的向量。

In [ ]:
# -- 将 50 轮对话送入向量存储记忆 --

vec_mem = VectorStoreMemory(
    top_k=5,
    recent_buffer_size=4,
    system_prompt="你是一个有帮助的助手。回复控制在 1-2 句话。",
    collection_name="experiment_vec",
)

print("正在将 50 轮对话送入向量存储记忆...")
for turn_num, msg in conversation_50:
    vec_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  第 {turn_num}/50 轮完成")

print(f"\n\u2713 完成。已存储向量: {vec_mem.collection.count()}")

现在测试回忆。对 12 条植入事实分别提问，检查回答中是否包含预期关键词。同时记录向量存储检索到的轮次，以验证是否找到了正确的来源。

In [ ]:
# -- 回忆测试：向量存储记忆 --

print("=== 向量存储记忆 — 回忆测试 ===\n")
vec_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    # 检查检索到了什么
    retrieved = vec_mem.search(question, top_k=5)
    retrieved_turns = [r["turn"] for r in retrieved]

    answer = vec_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    vec_results.append({
        "question": question,
        "keyword": keyword,
        "planted_at": planted_at,
        "recalled": recalled,
        "retrieved_turns": retrieved_turns,
        "answer": answer,
    })

    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (植入轮次 {planted_at:2d}) {question}")
    print(f"    检索到的轮次: {retrieved_turns}")
    print(f"    回答: {answer[:100]}")
    print()

vec_score = sum(1 for r in vec_results if r["recalled"])
print(f"向量存储记忆得分: {vec_score}/{len(RECALL_QUESTIONS)}")

### 基线：滑动窗口记忆（K=10）

作为对比，我们将相同的 50 轮对话送入一个只保留最近 10 条消息（5 轮用户-助手交换）的滑动窗口。这是公平的比较：两个系统每次 LLM 调用使用的上下文量大致相同。

In [ ]:
class SlidingWindowMemory:
    """简单的滑动窗口：只保留最近 K 条消息。"""

    def __init__(self, window_size: int = 10, model: str = "claude-sonnet-4-20250514",
                 system_prompt: str | None = None, max_tokens: int = 1024):
        self.window_size = window_size
        self.client = anthropic.Anthropic()
        self.model = model
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.messages: list[dict] = []

    def chat(self, user_input: str) -> str:
        self.messages.append({"role": "user", "content": user_input})

        # 只保留最近 window_size 条消息
        window = self.messages[-self.window_size:]

        kwargs = dict(model=self.model, max_tokens=self.max_tokens, messages=window)
        if self.system_prompt:
            kwargs["system"] = self.system_prompt

        response = self.client.messages.create(**kwargs)
        assistant_text = response.content[0].text

        self.messages.append({"role": "assistant", "content": assistant_text})
        return assistant_text


print("\u2713 SlidingWindowMemory 类已定义")

将相同的 50 轮对话送入滑动窗口基线。窗口只保留最近 10 条消息（5 轮交换），因此较早的事实会离开上下文。

In [ ]:
# -- 将相同的 50 轮对话送入滑动窗口 --

sw_mem = SlidingWindowMemory(
    window_size=10,
    system_prompt="你是一个有帮助的助手。回复控制在 1-2 句话。",
)

print("正在将 50 轮对话送入滑动窗口记忆...")
for turn_num, msg in conversation_50:
    sw_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  第 {turn_num}/50 轮完成")

print(f"\n\u2713 完成。窗口保留最近 {sw_mem.window_size} 条消息。")

对滑动窗口运行相同的回忆问题。由于它只能看到最近 10 条消息，早期轮次中早已离开窗口的事实应该难以回忆。

In [ ]:
# -- 回忆测试：滑动窗口记忆 --

print("=== 滑动窗口记忆 — 回忆测试 ===\n")
sw_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    answer = sw_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    sw_results.append({
        "question": question,
        "keyword": keyword,
        "planted_at": planted_at,
        "recalled": recalled,
        "answer": answer,
    })

    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (植入轮次 {planted_at:2d}) {question}")
    print(f"    回答: {answer[:100]}")
    print()

sw_score = sum(1 for r in sw_results if r["recalled"])
print(f"滑动窗口记忆得分: {sw_score}/{len(RECALL_QUESTIONS)}")

可视化结果。图 1 展示总回忆得分。图 2 按事实的年龄（植入轮次）分解回忆情况。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# -- 图 1：并排回忆得分 --
labels = ["向量存储\n记忆", "滑动窗口\n(K=10)"]
scores = [
    sum(1 for r in vec_results if r["recalled"]),
    sum(1 for r in sw_results if r["recalled"]),
]
colors = ["#4f46e5", "#ef4444"]
bars = axes[0].bar(labels, scores, color=colors, width=0.5, alpha=0.85)
axes[0].set_ylabel("回忆的事实数")
axes[0].set_ylim(0, len(RECALL_QUESTIONS) + 1)
axes[0].set_title("总回忆事实数（共 12 条）")
axes[0].axhline(y=len(RECALL_QUESTIONS), color="gray", linestyle="--", alpha=0.3)
for bar, score in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(score), ha="center", fontweight="bold", fontsize=14)

# -- 图 2：按轮次距离的回忆情况 --
fact_turns = [r["planted_at"] for r in vec_results]
vec_recalled = [1 if r["recalled"] else 0 for r in vec_results]
sw_recalled = [1 if r["recalled"] else 0 for r in sw_results]

x = np.arange(len(fact_turns))
width = 0.35
axes[1].bar(x - width/2, vec_recalled, width, label="向量存储", color="#4f46e5", alpha=0.85)
axes[1].bar(x + width/2, sw_recalled, width, label="滑动窗口", color="#ef4444", alpha=0.85)
axes[1].set_xlabel("事实植入轮次 #")
axes[1].set_ylabel("是否回忆？（1=是, 0=否）")
axes[1].set_title("按事实年龄的回忆情况")
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(t) for t in fact_turns], fontsize=8)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["否", "是"])
axes[1].legend()


图 3 检查检索精度。对每个回忆问题，向量存储是否检索到了事实最初植入的轮次？绿点表示是，橙色 X 表示正确轮次未出现在 Top-K 结果中。

In [ ]:
# -- 图 3：向量存储检索到了哪些轮次？ --
retrieved_hits = []
retrieved_misses = []
for r in vec_results:
    planted = r["planted_at"]
    retrieved = r.get("retrieved_turns", [])
    if planted in retrieved:
        retrieved_hits.append(planted)
    else:
        retrieved_misses.append(planted)

axes[2].scatter(retrieved_hits, [1]*len(retrieved_hits), color="#22c55e",
                s=120, marker="o", label="检索到正确轮次", zorder=3)
axes[2].scatter(retrieved_misses, [0]*len(retrieved_misses), color="#f59e0b",
                s=120, marker="x", label="未命中正确轮次", zorder=3)
axes[2].set_xlabel("事实植入轮次 #")
axes[2].set_ylabel("是否正确检索到轮次？")
axes[2].set_title("向量存储：检索到正确的轮次了吗？")
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(["否", "是"])
axes[2].legend(loc="center right")

plt.tight_layout()
plt.savefig("vector_vs_window.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n向量存储记忆: {scores[0]}/{len(RECALL_QUESTIONS)} 条回忆")
print(f"滑动窗口记忆: {scores[1]}/{len(RECALL_QUESTIONS)} 条回忆")

### 检索质量分析

深入查看向量存储为每个回忆问题检索到的*具体内容*。即使回答正确，了解哪些轮次被命中以及“正确”轮次是否在其中也很有价值。

In [ ]:
print("=== 检索分析 ===\n")
print(f"{'问题':<30} {'植入':>6} {'检索到':>20} {'命中?':>5}")
print("\u2500" * 75)

hits = 0
for r in vec_results:
    planted = r["planted_at"]
    retrieved = r.get("retrieved_turns", [])
    hit = planted in retrieved
    if hit:
        hits += 1
    print(f"{r['question'][:28]:<30} 第 {planted:<3}轮 {str(retrieved):<20} {'\u2713' if hit else '\u2717':>5}")

print(f"\n检索精度: {hits}/{len(vec_results)} 个查询检索到了正确的来源轮次。")

## 调优 Top-K：召回 vs. Token 成本

选择合适的 *K* 是典型的精度-召回权衡：
- **低 K**（1-2）：提示词中 token 更少，但可能遗漏相关上下文。
- **高 K**（10+）：更有可能包含正确记忆，但增加 token 成本，且可能引入无关噪音。

让我们在相同的 50 轮对话上测量检索命中率随 K 的变化。

In [ ]:
# 用已存储向量测试不同的 K 值
k_values = [1, 2, 3, 5, 8, 10, 15]
k_hit_rates = []

for k in k_values:
    hits = 0
    for question, keyword, planted_at in RECALL_QUESTIONS:
        retrieved = vec_mem.search(question, top_k=k)
        retrieved_turns = [r["turn"] for r in retrieved]
        if planted_at in retrieved_turns:
            hits += 1
    hit_rate = hits / len(RECALL_QUESTIONS)
    k_hit_rates.append(hit_rate)
    print(f"  K={k:2d}: {hits}/{len(RECALL_QUESTIONS)} 命中 ({hit_rate:.0%})")

# 绘图
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, k_hit_rates, "o-", color="#4f46e5", linewidth=2, markersize=8)
ax.fill_between(k_values, k_hit_rates, alpha=0.1, color="#4f46e5")
ax.set_xlabel("Top-K")
ax.set_ylabel("检索命中率")
ax.set_title("检索命中率 vs. Top-K")
ax.set_ylim(0, 1.05)
ax.set_xticks(k_values)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("topk_tuning.png", dpi=150, bbox_inches="tight")
plt.show()

比较三种记忆策略的 token 成本增长。完整缓冲每轮发送全部消息，所以成本线性增长。滑动窗口成本恒定但遗忘旧上下文。向量存储同样保持恒定，同时保留对所有历史轮次的访问。

In [ ]:
# 比较三种方案的 token 成本

MSG_TOKENS = 50   # 每条消息的平均 token 数
SYS_TOKENS = 30   # 系统提示词
K_RETRIEVE = 5    # Top-K 检索记忆
NUM_TURNS = 50

full_buffer, sliding_window, vector_store = [], [], []

for turn in range(1, NUM_TURNS + 1):
    n_msgs = turn * 2

    # 完整缓冲：全部消息
    full_buffer.append(SYS_TOKENS + n_msgs * MSG_TOKENS)

    # 滑动窗口（K=10 条消息）
    sliding_window.append(SYS_TOKENS + min(n_msgs, 10) * MSG_TOKENS)

    # 向量存储：K 条检索记忆 + 小型近期缓冲（4 条消息）
    retrieved_tokens = K_RETRIEVE * 2 * MSG_TOKENS  # 每条记忆含用户+助手
    recent_tokens = min(n_msgs, 4) * MSG_TOKENS
    vector_store.append(SYS_TOKENS + retrieved_tokens + recent_tokens)

turns = list(range(1, NUM_TURNS + 1))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(turns, full_buffer, "o-", color="#ef4444", label="完整缓冲", linewidth=2, markersize=3)
ax.plot(turns, sliding_window, "s-", color="#f59e0b", label="滑动窗口 (K=10)", linewidth=2, markersize=3)
ax.plot(turns, vector_store, "^-", color="#4f46e5", label="向量存储 (top-5 + 缓冲-4)", linewidth=2, markersize=3)

ax.set_xlabel("对话轮次")
ax.set_ylabel("每次 LLM 调用的输入 Token")
ax.set_title("每轮 Token 成本：三种记忆策略（50 轮）")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("token_cost_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"在第 50 轮:")
print(f"  完整缓冲:    {full_buffer[-1]:,} token/次")
print(f"  滑动窗口: {sliding_window[-1]:,} token/次")
print(f"  向量存储:   {vector_store[-1]:,} token/次")
print(f"\n向量存储保持成本恒定，同时维持完整回忆能力。")

## 讨论与权衡

### 优势
- **语义召回**：按*含义*而非位置检索上下文。第 3 轮的事实到第 50 轮仍和第 4 轮一样可访问。
- **有限的 Token 成本**：每轮成本大致恒定：`top_K x 分块大小 + 近期缓冲区`。无论对话多长都保持平稳。
- **可扩展**：向量数据库可处理数百万个向量。记忆可增长而不降低检索速度（通过 ANN 索引）。
- **可组合**：自然支持元数据过滤（时间范围、主题、用户 ID），实现更有针对性的检索。

### 不足
- **基础设施开销**：需要嵌入模型和向量数据库，增加了复杂度、延迟和每轮成本。
- **嵌入质量瓶颈**：如果嵌入模型在特定领域语义上表现不佳，检索将不可靠。
- **缺乏叙事连贯性**：检索到的记忆是孤立片段，而非连续叙事。LLM 必须从分散的片段中重建连贯性。
- **语义相关不等于上下文相关**：两段文字可能语义相似但上下文无关（如“我喜欢 Python”和“Python 是一种蛇”）。
- **冷启动**：存储的记忆较少时，检索噪音大。系统随记忆增长而改善。

### 何时使用向量存储记忆

| 场景 | 建议 |
|------|------|
| 长对话（50+ 轮），用户会回溯早期话题 | 适合，理想场景 |
| 需要回忆之前对话的多会话 Agent | 适合，跨会话嵌入 |
| 每毫秒延迟都至关重要的实时聊天 | 谨慎，嵌入增加约 50-100ms 延迟 |
| 短对话（< 10 轮） | 杀鸡用牛刀，滑动窗口足够 |
| 需要精确的时间叙事 | 不适合，用缓冲或摘要代替 |
| 混合：回忆旧事实 + 理解近期流程 | 适合，向量存储配合近期缓冲区（如本实现） |

### 成本模型
每轮：
- **1 次嵌入调用** 用于用户查询（`text-embedding-3-small` 约 $0.00002）
- **1 次嵌入调用** 用于存储交换（约 $0.00002）
- **1 次向量搜索**（ChromaDB 内存模式免费，约 10ms）
- **1 次 LLM 调用**，使用 `top_K x 分块大小 + 缓冲区大小` token

嵌入成本相比 LLM 调用可忽略不计。真正的节省来自*不*发送整个对话历史。

## 延伸阅读

- [ChromaDB 文档](https://docs.trychroma.com/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：本 notebook 使用的内存向量数据库
- [OpenAI 嵌入指南](https://platform.openai.com/docs/guides/embeddings?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：text-embedding-3-small 的工作原理和定价
- [LangChain VectorStoreRetrieverMemory](https://python.langchain.com/docs/modules/memory/types/vectorstore_retriever_memory?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：框架级集成
- [Pinecone：生产级向量数据库](https://www.pinecone.io/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：托管向量数据库替代方案
- [FAISS：Facebook AI 相似度搜索](https://github.com/facebookresearch/faiss)：高性能本地向量搜索
- [Anthropic：构建对话式 AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：多轮对话模式
- [Lilian Weng, "LLM Powered Autonomous Agents"](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：Memory 部分涵盖向量检索模式

---

*← 上一章：[05：Token 缓冲记忆](../05_token_buffer_memory/) · 下一章：[07：实体记忆](../07_entity_memory/) →*

In [ ]:
# 清理演示中创建的临时文件
import os
for f in ["vector_vs_window.png", "topk_tuning.png", "token_cost_comparison.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"已删除 {f}")

## 🧪 自己动手试试

三个小挑战来加深你的理解。每个应该花费 10-30 分钟。

### 挑战 1：嵌入模型对比
在 `_embed()` 中将 `text-embedding-3-small` 替换为 `text-embedding-3-large`。运行相同的 50 轮实验，比较 K=5 时的命中率。记录每个模型的嵌入延迟，观察速度与质量的权衡。

### 挑战 2：K 参数扫描
使用 K 值 1、3、5、10、20 运行检索。对每个 K，使用回忆评估测量事实保留率。绘制 K vs. 召回率和 K vs. 每轮平均输入 token。找到每花一个 token 能获得最大召回率的 K 值。

### 挑战 3：元数据过滤检索
为每条存储的交换添加 `turn_number` 元数据字段。实现一个按近因过滤的 `_retrieve()` 变体（只在最近 N 轮中进行向量搜索）。将这种时间限定的向量搜索与纯语义检索进行对比，借鉴 18 时间记忆的思路。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--06-vector-store-memory--vector-store-memory)
